# FolkFusion — Colab Model Server

This runs Stable Diffusion img2img on a free T4 GPU and exposes it as a public API that your local Flask app calls.

**Steps:**
1. Make sure Runtime → Change runtime type → T4 GPU is selected
2. Run Cell 1 (install, ~2 min)
3. Run Cell 2 (load model, ~3 min)
4. Copy the `gradio.live` URL printed at the end
5. Paste it into your `.env` as `COLAB_URL=https://xxxx.gradio.live`
6. Restart your Flask app

In [ ]:
# Cell 1 — Install dependencies (~2 min)
!pip install -q diffusers transformers accelerate gradio torch

In [ ]:
# Cell 2 — Load model and start server (~3 min first time)
import torch
from diffusers import StableDiffusionImg2ImgPipeline
from PIL import Image
import gradio as gr

print('Loading SD img2img model...')
pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    torch_dtype=torch.float16,
    safety_checker=None,
)
pipe = pipe.to('cuda')
print('Model loaded!')

DEFAULT_NEGATIVE = (
    'photorealistic, photography, 3d render, western oil painting, '
    'acrylic, watercolor, pencil sketch, digital art, modern illustration, '
    'blurry, low quality, deformed, noisy, distorted'
)

def transform(image, prompt, strength, guidance, steps):
    result = pipe(
        prompt,
        image=image,
        negative_prompt=DEFAULT_NEGATIVE,
        num_inference_steps=int(steps),
        strength=float(strength),
        guidance_scale=float(guidance),
    ).images[0]
    return result

iface = gr.Interface(
    fn=transform,
    inputs=[
        gr.Image(type='pil', label='Input image'),
        gr.Textbox(label='Style prompt'),
        gr.Slider(0.3, 0.9, value=0.65, step=0.05, label='Strength (higher = more stylized)'),
        gr.Slider(5.0, 20.0, value=12.0, step=0.5, label='Guidance scale'),
        gr.Slider(20, 60, value=40, step=1, label='Steps'),
    ],
    outputs=gr.Image(type='pil', label='Result'),
    title='FolkFusion Model Server',
)

iface.launch(share=True)